# Explaining a financing optimiser through a surrogate model

The Pyomo model computes the minimum total debt an electrification project in Rwanda needs, given how its financing is negotiated. It's an optimiser, not a learned model, so explainability techniques can't be applied to it directly. Instead, I sample the negotiation parameters, solve Pyomo for each combination, and train a surrogate regressor on the results. The explanations are computed on the surrogate, but since I still have the real model, I can check whether they're actually right.

## 1. Problem and synthetic dataset

### 1.1 The synthetic dataset

The dataset was generated with `scripts/run_01_generate_dataset.py`. The eight inputs are degrees of freedom of the negotiation (capital structure, debt terms, grants and tariff), each one controlled by a different actor. Every technical input (capex, demand, opex, year horizon) stays fixed at the Rwanda case.

Since these inputs are negotiated rather than observed, there's no real distribution to estimate, so each one was sampled uniformly over the plausible range agreed with a supervisor. Uniform is the maximum-entropy choice when only the bounds are known: it adds no assumption beyond them. I deliberately avoided a unimodal distribution centred on typical values. That would put most samples around the usual deals, so the surrogate would learn that region best and the explanations would mostly describe the status quo. What I want to know is how much each actor *could* move the debt if they pushed their parameter anywhere in its negotiable range, and that needs every outcome, typical or extreme, to carry the same weight. How the picture changes for typical negotiations is analysed separately later on.

I used Latin Hypercube Sampling so that every input covers its whole range evenly, and solved Pyomo for 6000 combinations. All of them reached a proven optimum.

Besides the eight inputs and the total debt, each sample stores two kinds of extra columns. The first are generation checks (solver status, the largest share of the big-M bound reached and the solve time), which just confirm the data is valid. The second are the totals of each funding source over the horizon: grants from the DFI, long-term subsidies from the government, the service margin paid by consumers through the tariff, and the equity put in by the sponsors. None of these go into the surrogate. They are outputs of the same optimisation, so feeding them in would leak the answer. I keep them to add context to the explanations later on: an actor can lower the debt by shifting the cost onto someone else, and these totals show who ends up paying. One caveat: total debt is the only quantity the optimiser pins down uniquely, so the other totals are descriptive rather than exact.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
DATA_PATH = PROJECT_ROOT / 'data' / 'dataset_uniform_n6000_seed42.csv'

# Only the eight negotiation inputs feed the surrogate
FEATURES = ['pct_debt', 'cost_debt', 'equity_premium', 'grace_period', 'amortization_period', 'pct_grants', 'years_realisation', 'tariff_y1']
TARGET = 'total_debt'

dataset = pd.read_csv(DATA_PATH)
X, y = dataset[FEATURES], dataset[TARGET]
print(X.shape, y.shape)

(6000, 8) (6000,)


In [2]:
print(f"Zero-debt samples: {(y == 0).sum()} / {len(y)} ({(y == 0).mean():.1%})")

Zero-debt samples: 2239 / 6000 (37.3%)


### 1.2 Train, validation and test split

About 37% of the samples need no debt at all: under favourable terms the project funds itself. Because that's such a big chunk of the data, I stratify the split on zero versus positive debt, so the three subsets keep the same proportion and their metrics are comparable. Train is used to fit the surrogate, validation to tune it, and test stays untouched until the final evaluation.

In [3]:
is_zero_debt = (y == 0)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=is_zero_debt, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=is_zero_debt.loc[y_temp.index], random_state=42)

for name, target in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(f"{name:<5} n={len(target):4d}  zero-debt share={(target == 0).mean():.1%}")

train n=4200  zero-debt share=37.3%
val   n= 900  zero-debt share=37.3%
test  n= 900  zero-debt share=37.3%


### 1.3 When does the project need no debt?

Before any modelling, a quick look at which negotiation conditions lead to zero debt. Since the inputs were sampled independently, comparing their averages between debt-free and indebted samples gives a fair first idea of what drives the project into self-financing, without one input masking another.

In [4]:
zero_debt = dataset[TARGET] == 0
comparison = pd.DataFrame({
    'zero debt':     dataset.loc[zero_debt, FEATURES].mean(),
    'positive debt': dataset.loc[~zero_debt, FEATURES].mean(),
})
comparison['difference (%)'] = 100 * (comparison['zero debt'] / comparison['positive debt'] - 1)
comparison.round(3)

,zero debt,positive debt,difference (%)
pct_debt,0.588,0.607,-3.096
cost_debt,0.089,0.082,8.400
equity_premium,0.083,0.078,6.347
grace_period,2.536,2.479,2.314
amortization_period,20.086,19.949,0.686
pct_grants,0.271,0.238,13.779
years_realisation,3.142,2.915,7.772
tariff_y1,0.216,0.135,60.149


The tariff stands out by far: debt-free projects have an average first-year tariff of 0.216 against 0.135 for indebted ones, 60% higher. This fits the mechanism in the model, where a tariff above the cost of service turns into margin, which is cash. Grants help too, but much less (+14%), a bit more when they arrive over more years (+8% in realisation years). Grace and amortisation periods barely differ between the groups, so at this level they look irrelevant.

The surprising part is that the cost of debt (+8%) and the equity premium (+6%) are *higher* in the debt-free group, when you'd expect expensive capital to push the project into more borrowing. My guess is that it comes from cost-of-service regulation: the allowed return is the WACC on the asset base, so more expensive capital raises the recognised cost of service, and that extra cost is passed on to whoever pays it (consumers or the government). I'll check this hypothesis directly against Pyomo later. Comparing averages only shows broad tendencies though. It can't reveal thresholds or interactions, which is what the explainability methods are for.

## 2. Surrogate model

The surrogate maps the eight negotiation inputs to a single number, the total debt over the 12 years, so this is a tabular regression problem rather than a time-series one. There is a time dimension inside Pyomo (debt, losses and the asset base carry over from one year to the next), but it stays inside the optimiser. Each row of the dataset is an independent combination of inputs with its resulting debt, and the order of the rows means nothing. Time-series models like ARIMA also wouldn't fit the year-by-year structure itself. They explain each value from the series' own past plus random noise, whereas here the trajectory is fully determined by the inputs, and since Pyomo optimises all 12 years at once, early decisions already take later years into account.

### 2.1 Linear baseline

Before going for a black-box model, I fit a plain linear regression. It's the textbook transparent model, since its coefficients are the explanation, so it sets the bar: if a more complex surrogate can't clearly beat it, the extra opacity isn't worth it. I compare models with MAE, RMSE and R² on the validation set.

In [5]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def report(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name:<28} MAE={mae:7.2f}  RMSE={rmse:7.2f}  R²={r2:.3f}")


linear = LinearRegression().fit(X_train, y_train)
report('Linear regression (val)', y_val, linear.predict(X_val))

Linear regression (val)      MAE= 150.70  RMSE= 207.65  R²=0.641


The linear model captures the general trend (R² = 0.64) but its errors are large: an average miss of about 151 when the mean debt is around 248, so roughly 60% of a typical value. It also has no way to represent the floor at zero debt that more than a third of the samples sit on. A straight line can't go flat at zero and then start rising, so it ends up predicting debt where there is none and the other way round. A non-linear surrogate is clearly needed.

### 2.2 Gradient boosting surrogate

Gradient boosting builds a sequence of small decision trees, each one fitted to the errors left by the previous ones. Trees split the input space with thresholds, which is exactly what this problem needs: debt sits at a floor of zero over a big part of the space, and the tariff only matters once it goes above the cost of service. Boosting also picks up interactions between inputs without me specifying them. Last, tree models allow TreeSHAP, which computes exact Shapley values instead of a sampling approximation, so any mismatch I find later against Pyomo comes from the surrogate and not from the explainer.

In [6]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(random_state=42).fit(X_train, y_train)
report('Gradient boosting (val)', y_val, gbr.predict(X_val))

Gradient boosting (val)      MAE=  44.29  RMSE=  67.57  R²=0.962


With default settings, gradient boosting already cuts the validation MAE to under a third of the linear model's (44 against 151) and pushes R² from 0.64 to 0.962, which confirms the problem is strongly non-linear. An average error of around 44 is still about 18% of the mean debt though, so it's worth tuning the model before trusting its explanations.

### 2.3 Hyperparameter tuning

I tune the three settings that matter most for boosting: the number of trees, their depth and the learning rate. Each combination is fitted on the training set and scored on the validation set. The test set stays untouched, so the final score remains an honest estimate.

In [7]:
from itertools import product

grid = {
    'n_estimators':  [200, 500, 1000],
    'max_depth':     [3, 4, 5],
    'learning_rate': [0.05, 0.1],
}

results = []
for n_est, depth, lr in product(*grid.values()):
    model = GradientBoostingRegressor(
        n_estimators=n_est, max_depth=depth, learning_rate=lr, random_state=42
    ).fit(X_train, y_train)
    val_pred = model.predict(X_val)
    results.append({'n_estimators': n_est, 'max_depth': depth, 'learning_rate': lr,
                    'val_MAE': mean_absolute_error(y_val, val_pred),
                    'val_RMSE': np.sqrt(mean_squared_error(y_val, val_pred))})

tuning = pd.DataFrame(results).sort_values('val_RMSE')
tuning.head(5).round(2)

,n_estimators,max_depth,learning_rate,val_MAE,val_RMSE
16,1000,5,0.05,24.18,40.68
17,1000,5,0.10,24.90,41.10
10,500,5,0.05,24.56,41.34
11,500,5,0.10,24.90,41.34
5,200,5,0.10,25.73,42.80


With the larger dataset the best grid combination reaches a validation MAE of 24.2 and RMSE of 40.7. All five top configurations use depth 5, the deepest value in the grid, so the optimum may lie beyond it. That makes sense with more data: deeper trees can capture more complex interactions without overfitting. The next search therefore moves the depth range up to 5–6 and adds row subsampling.

In [8]:
# One fit with 6000 trees per setting; staged_predict gives the validation error after each tree
for depth, lr in product([5, 6], [0.02, 0.05]):
    model = GradientBoostingRegressor(
        n_estimators=6000, max_depth=depth, learning_rate=lr, subsample=0.8, random_state=42
    ).fit(X_train, y_train)
    val_rmse = [np.sqrt(mean_squared_error(y_val, pred)) for pred in model.staged_predict(X_val)]
    best_n = int(np.argmin(val_rmse)) + 1
    print(f"depth={depth}  lr={lr}  RMSE at 3000 trees={val_rmse[2999]:.2f}  "
          f"best n_estimators={best_n:4d}  best RMSE={min(val_rmse):.2f}")

depth=5  lr=0.02  RMSE at 3000 trees=35.36  best n_estimators=6000  best RMSE=35.14
depth=5  lr=0.05  RMSE at 3000 trees=37.02  best n_estimators=5983  best RMSE=36.99
depth=6  lr=0.02  RMSE at 3000 trees=36.55  best n_estimators=5921  best RMSE=36.48
depth=6  lr=0.05  RMSE at 3000 trees=39.64  best n_estimators=5816  best RMSE=39.64


Adding row subsampling (each tree sees a random 80% of the training data, i.e. stochastic gradient boosting) and letting the model grow more trees brings the validation RMSE down to the 35–40 range. Depth 6 is worse than depth 5 at both learning rates, so depth 5 is an interior optimum rather than the edge of the search. To pick the number of trees I fitted each setting with 6000 trees and read the validation error after each one. The best values all land near 6000, but the curves are flat by then: for depth 5 and learning rate 0.02, going from 3000 to 6000 trees only lowers the RMSE from 35.36 to 35.14. I keep depth 5, learning rate 0.02 and 3000 trees: it is the most accurate configuration, and more trees would only slow down SHAP for no real gain.

### 2.4 Final surrogate and test performance

With the hyperparameters fixed, I refit both models on train and validation together, since validation has done its job and more data gives a better surrogate. Then I score them once on the untouched test set.

In [11]:
best_params = dict(n_estimators=3000, max_depth=5, learning_rate=0.02, subsample=0.8, random_state=42)

X_trval = pd.concat([X_train, X_val])
y_trval = pd.concat([y_train, y_val])

surrogate = GradientBoostingRegressor(**best_params).fit(X_trval, y_trval)
linear_final = LinearRegression().fit(X_trval, y_trval)

report('Linear regression (test)', y_test, linear_final.predict(X_test))
report('Gradient boosting (test)', y_test, surrogate.predict(X_test))

Linear regression (test)     MAE= 154.46  RMSE= 211.58  R²=0.623
Gradient boosting (test)     MAE=  19.08  RMSE=  31.52  R²=0.992


On the test set the gradient boosting surrogate reaches R² = 0.992 with an MAE of about 19, under 8% of the mean debt and eight times lower than the linear model. Its test RMSE (31.5) is even a bit lower than the validation one (35.4). That's expected, since the final model was refitted on train and validation together, so it learned from more data. It also confirms that tuning didn't overfit the validation set. The RMSE is still about 1.7 times the MAE though, which means most predictions are very accurate but a handful of samples carry larger errors. I look at where those are next, since the explanations will be less reliable in those regions.

### 2.5 Is the dataset large enough?

A first version of this analysis used 3000 samples, and its learning curve was still falling steeply at the full training size, so I regenerated the dataset with 6000. Here I check whether that was enough: I refit the final configuration on growing training sizes, from about the size of the old training set (2100) up to the full 4200, and track the validation error. If the curve flattens beyond 2100, doubling the data paid off and more samples wouldn't help much.

In [12]:
# Learning curve: the split is already shuffled, so the first n rows are a random subset.
# Sizes span the old 3000-sample dataset (~2100 training rows) and beyond.
for n in [525, 1050, 2100, 3150, 4200]:
    model = GradientBoostingRegressor(**best_params).fit(X_train.iloc[:n], y_train.iloc[:n])
    val_pred = model.predict(X_val)
    print(f"n_train={n:4d}  val MAE={mean_absolute_error(y_val, val_pred):6.2f}  "
          f"val RMSE={np.sqrt(mean_squared_error(y_val, val_pred)):6.2f}")

n_train= 525  val MAE= 39.65  val RMSE= 71.05
n_train=1050  val MAE= 33.41  val RMSE= 60.16
n_train=2100  val MAE= 25.68  val RMSE= 45.89
n_train=3150  val MAE= 22.85  val RMSE= 39.28
n_train=4200  val MAE= 21.03  val RMSE= 35.36


Doubling the data clearly paid off: at the old training size (2100) the validation RMSE was about 46, and with the full 4200 it drops to about 35. Beyond 2100 the gains shrink, from 6.6 points in the next step to 3.9 in the last one, so the curve is flattening, although it isn't completely flat yet. Another doubling would probably shave off a few more points at the cost of roughly another hour of Pyomo solves. With a test R² of 0.992 I stop here. Whether the remaining error actually distorts the explanations is checked directly against Pyomo in Section 4, which is a better basis for deciding than piling up more data blindly.

Surrogate año a año (trabajo futuro, descartado por ahora)

Idea: en vez de predecir la deuda total, predecir la deuda de cada uno de los 12 años, con una regresión multi-salida (8 entradas → 12 salidas). No sería un modelo de series temporales tipo ARIMA, porque la trayectoria la determinan las entradas y Pyomo optimiza los 12 años a la vez.
Por qué no ahora:
El reparto de la deuda entre años puede no ser único. El objetivo minimiza la suma de la deuda, así que si dos calendarios dan la misma suma, HiGHS elige uno cualquiera. El total es un óptimo demostrado, pero el reparto anual puede ser arbitrario, y el surrogate aprendería esa arbitrariedad.
Multiplica por 12 el trabajo de XAI (SHAP, PDP, comprobaciones contra Pyomo) y la rúbrica no lo pide.
La pregunta del proyecto es sobre el total: qué actor tiene más poder sobre la deuda final.
Qué haría falta para hacerlo bien: un objetivo que fije el calendario de la deuda (por ejemplo, penalizar también la deuda máxima de un año o el momento en que se pide). Así el reparto anual sería único y explicable.
Dónde mencionarlo: como limitación y trabajo futuro en el paper (reflexión crítica).